# MCP

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# =============================================================================
# Windows + Jupyter + MCP Fix: Async Subprocess Support
# =============================================================================
# Problem: Windows SelectorEventLoop doesn't support subprocess creation
# Solution: Replace the running event loop with ProactorEventLoop
# =============================================================================

if sys.platform == "win32":
    # Step 1: Set ProactorEventLoop policy for subprocess support
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # Step 2: Close and replace Jupyter's existing event loop
    # Jupyter creates a SelectorEventLoop before our code runs, so we must replace it
    try:
        loop = asyncio.get_running_loop()
        # We can't replace a running loop, but we can verify it's the right type
        if not isinstance(loop, asyncio.ProactorEventLoop):
            print("WARNING: Event loop is already running and is not ProactorEventLoop")
            print("You may need to restart the kernel for full compatibility")
    except RuntimeError:
        # No loop is running, create a new ProactorEventLoop
        loop = asyncio.ProactorEventLoop()
        asyncio.set_event_loop(loop)

    # Step 3: Fix stderr for subprocess compatibility
    # Jupyter's sys.stderr doesn't have fileno(), which breaks subprocess.Popen
    if "ipykernel" in sys.modules:
        import io
        # Replace Jupyter's stderr with the original stderr or a null stream
        if hasattr(sys, '__stderr__') and sys.__stderr__ is not None:
            sys.stderr = sys.__stderr__
        else:
            # Fallback: use devnull if __stderr__ is not available
            sys.stderr = open('nul', 'w')  # 'nul' is Windows equivalent of /dev/null

print(f"Event loop type: {type(asyncio.get_event_loop()).__name__}")
print(f"Event loop policy: {type(asyncio.get_event_loop_policy()).__name__}")
print("Windows MCP subprocess support configured successfully!")

You may need to restart the kernel for full compatibility
Event loop type: _WindowsSelectorEventLoop
Event loop policy: WindowsProactorEventLoopPolicy
Windows MCP subprocess support configured successfully!


# Local MCP Server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command" : "python",
            "args": ["resources/2.1_mcp_server.py"]
        }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [11]:
len(tools)

1

In [12]:
prompt

'\n    You are a helpful assistant that answers user questions about LangChain, LangGraph and LangSmith.\n\n    You can use the following tools/resources to answer user questions:\n    - search_web: Search the web for information\n    - github_file: Access the langchain-ai repo files\n\n    If the user asks a question that is not related to LangChain, LangGraph or LangSmith, you should say "I\'m sorry, I can only answer questions about LangChain, LangGraph and LangSmith."\n\n    You may try multiple tool and resource calls to answer the user\'s question.\n\n    You may also ask clarifying questions to the user to better understand their question.\n    '

In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [14]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [15]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='909094c8-80b7-4c1d-8ffd-54523c01c85e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 271, 'total_tokens': 363, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D6z41dMta5OOaEBWSxZUYpU6l2w6g', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c3d6c-d504-7092-b936-8cd8dca0251e-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters'}, 'id': 'call_iuJgrg5KmF2lB4Fr3bw7XpfS', 'type': 'tool_call'}], invalid_tool_calls=[], usag

In [16]:
pprint(response['messages'][-1].content)

('Here’s a concise overview of the langchain-mcp-adapters library and what '
 'it’s for.\n'
 '\n'
 'What it is\n'
 '- A bridge that lets you use tools exposed via the Anthropic Model Context '
 'Protocol (MCP) inside LangChain and LangGraph.\n'
 '- It converts MCP tools into LangChain- and LangGraph-compatible tools, and '
 'supports interacting with tools across multiple MCP servers.\n'
 '- It also helps integrate MCP tool ecosystems into LangGraph agents, making '
 'it easy to pull from many servers at once.\n'
 '\n'
 'What you can do with it\n'
 '- Load tools from one or more MCP servers without writing custom adapters.\n'
 '- Use MCP tools inside LangChain agents and LangGraph workflows.\n'
 '- Load tools from multiple servers and combine them for richer tool use.\n'
 '- Convert MCP prompt messages into LangChain message objects (so MCP prompts '
 'can flow naturally in LangChain/LangGraph).\n'
 '\n'
 'Core components (high level)\n'
 '- Python side (langchain_mcp_adapters):\n'
 ' 

In [17]:
print(type(response['messages']))

<class 'list'>


In [18]:
len(response['messages'])

4

# Importing an MCP from mcp.so
- Link - https://mcp.so/servers

In [19]:
client = MultiServerMCPClient(
    {
        "time":{
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [20]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools
)

In [23]:
tools

[StructuredTool(name='get_current_time', description='Get current time in a specific timezones', args_schema={'type': 'object', 'properties': {'timezone': {'type': 'string', 'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no timezone provided by the user."}}, 'required': ['timezone']}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000001C370672B60>),
 StructuredTool(name='convert_time', description='Convert time between timezones', args_schema={'type': 'object', 'properties': {'source_timezone': {'type': 'string', 'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no source timezone provided by the user."}, 'time': {'type': 'string', 'description': 'Time to convert in 24-hour format (HH:MM)'}, 'target_timezone': {'type': 'string', 'description': "Target IANA ti

In [24]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='a009b28d-c604-4aad-b183-d7b4f1153add'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 219, 'prompt_tokens': 296, 'total_tokens': 515, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D72UNq7seMUNep9Y8O0Yw8blpLb3K', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c3e35-be55-7fa0-8ba5-a3db1ed4e9c4-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'call_y5pkP9ytsRrLMh5rpHCdRZCm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens':

In [25]:
pprint(response['messages'][-1].content)

('Right now in New York: Sunday, February 8, 2026, 12:03:59 PM (EST). Would '
 'you like me to convert this to another timezone?')


# Kiwi Travel MCP

In [27]:
from langchain_mcp_adapters.client import  MultiServerMCPClient

kiwiClient = MultiServerMCPClient(
    {
        "travel_server":{
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    }
)

tools = await kiwiClient.get_tools()

pprint(tools)

[StructuredTool(name='search-flight', description='\n# Search for a flight\n\n## Description\n\nUses the Kiwi API to search for available flights between two locations on a specific date.\n\n## How it works\n\nThe tool will:\n1. Search for matching locations to resolve airport codes\n2. Find available flights for the specified route and date range\n\n## Method\n\nCall this tool whenever a user wants to search for flights, regardless of whether they provided exact airport codes or just city names.\n\nYou should display the returned results in a markdown table format: Group the results by price (those who are the cheapest), duration (those who are the shortest, i.e. have the smallest \'totalDurationInSeconds\') and the rest (those that could still be interesting).\n\nAlways display for each flight in order:\n  - In the 1st column: The departure and arrival airports, including layovers (e.g. "Paris CDG → Barcelona BCN → Lisbon LIS")\n  - In the 2nd column: The departure and arrival dates 

In [28]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow-up questions."
)

In [32]:
from langchain.messages import HumanMessage

config: {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Give me flights from Bengaluru to Delhi on February 13th")]},
    config
)

In [33]:
pprint(response)

{'messages': [HumanMessage(content='Give me a direct flight from Bengaluru to Delhi on February 13th', additional_kwargs={}, response_metadata={}, id='d0d960ad-2193-4a1c-ba4e-a6394650151f'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 833, 'prompt_tokens': 1228, 'total_tokens': 2061, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D72cXPFBB5uFULfFxGVsw7CHjbU1a', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c3e3d-78b4-7f20-ad13-3106f51c641d-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'Bengaluru', 'flyTo': 'Delhi', 'departureDate': '13/02/2026', 'passengers': {'adults': 1}, 

In [34]:
print(response['messages'][-1].content)

Here are the flight options from Bengaluru (BLR) to Delhi (DEL) for February 13, 2026. Results grouped by cheapest, then shortest duration, then other interesting options.

Cheapest flights
| Route | Times (Local) | Cabin | Return Route | Return Times | Return Cabin | Price | Book |
|---|---|---|---|---|---|---:|---|
| BLR → BOM → DEL (1 stop) | 13/02 07:40 → 13/02 10:50 (3h10m) | Economy |  |  |  | INR 7,467 | https://on.kiwi.com/zDzAiZ |

Shortest duration flights
| Route | Times (Local) | Cabin | Return Route | Return Times | Return Cabin | Price | Book |
|---|---|---|---|---|---|---:|---|
| BLR → DEL (nonstop) | 13/02 07:00 → 13/02 09:45 (2h45m) | Economy |  |  |  | INR 8,223 | https://on.kiwi.com/TW1YAq |

Other interesting options
| Route | Times (Local) | Cabin | Return Route | Return Times | Return Cabin | Price | Book |
|---|---|---|---|---|---|---:|---|
| BLR → DEL (nonstop) | 13/02 00:15 → 13/02 03:15 (3h) | Economy |  |  |  | INR 7,607 | https://on.kiwi.com/g47MTG |
| BLR →

In [ ]:
# EOF